# RQ1 — Notebook 3: Method Redundancy Analysis

**Research Question**: Are any of the 6 clustering methods redundant, and which can be removed?

This notebook evaluates redundancy on the current **94-row** LSE table from notebook 02. Three analyses are run:

1. **6×6 Pearson correlation matrix** of LSE values across datasets
2. **Wins distribution** showing how often each method is the true argmax-best
3. **Marginal contribution to oracle** showing the mean oracle LSE drop if a method is removed

**Current decision**: all six methods are retained. No method fails both retention thresholds.

**Outputs**: `outputs/figures/method_correlation.png`, `method_wins.png`, `method_marginal.png`, and `data/meta_table/method_redundancy_summary.csv`


In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT     = os.path.abspath(os.path.join(os.getcwd(), '..'))
META_DIR = os.path.join(ROOT, 'data', 'meta_table')
FIGS_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

LSE_COLS     = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']
METHOD_NAMES = ['kmeans', 'dbscan', 'agg', 'gmm', 'autoenc', 'dictlearn']

df = pd.read_csv(os.path.join(META_DIR, 'meta_training.csv'))
print(f'Loaded meta_training.csv: {df.shape}')
print(f'Datasets: {len(df)}')

Loaded meta_training.csv: (94, 9)
Datasets: 94


## Analysis 1: Method Correlation Matrix

Pearson correlation of LSE values across all 94 datasets. The strongest pair in the current run is **GMM vs autoencoder (r = 0.883)**, followed by k-means vs GMM (r = 0.850), k-means vs autoencoder (r = 0.837), and k-means vs agglomerative (r = 0.829). These correlations are high but not sufficient by themselves to remove a method.


In [2]:
corr_matrix = df[LSE_COLS].corr(method='pearson')
corr_matrix.index   = METHOD_NAMES
corr_matrix.columns = METHOD_NAMES

print('=== 6x6 Pearson Correlation of LSE values ===')
print(corr_matrix.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, ax=ax,
    mask=np.zeros_like(corr_matrix, dtype=bool),  # show full matrix
)
ax.set_title('LSE Correlation Between Methods\n(r > 0.85 = potentially redundant)', fontsize=11)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_correlation.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

# Highlight high-correlation pairs
print('\n=== High-correlation pairs (|r| > 0.70) ===')
for i in range(len(METHOD_NAMES)):
    for j in range(i+1, len(METHOD_NAMES)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.70:
            print(f'  {METHOD_NAMES[i]:10s} vs {METHOD_NAMES[j]:10s}  r={r:.3f}')

=== 6x6 Pearson Correlation of LSE values ===
           kmeans  dbscan    agg    gmm  autoenc  dictlearn
kmeans      1.000   0.643  0.829  0.850    0.837      0.721
dbscan      0.643   1.000  0.651  0.616    0.620      0.726
agg         0.829   0.651  1.000  0.800    0.762      0.691
gmm         0.850   0.616  0.800  1.000    0.883      0.619
autoenc     0.837   0.620  0.762  0.883    1.000      0.606
dictlearn   0.721   0.726  0.691  0.619    0.606      1.000
Saved → c:\MLResearch\outputs\figures\method_correlation.png

=== High-correlation pairs (|r| > 0.70) ===
  kmeans     vs agg         r=0.829
  kmeans     vs gmm         r=0.850
  kmeans     vs autoenc     r=0.837
  kmeans     vs dictlearn   r=0.721
  dbscan     vs dictlearn   r=0.726
  agg        vs gmm         r=0.800
  agg        vs autoenc     r=0.762
  gmm        vs autoenc     r=0.883


## Analysis 2: Wins Distribution

For each method, count how many datasets have it as the argmax-best. In the current run, every method wins at least 5 datasets: k-means 28, GMM 21, agglomerative 17, autoencoder 13, dictionary learning 10, and DBSCAN 5.


In [3]:
wins = df['best_method'].value_counts().reindex(METHOD_NAMES, fill_value=0)
wins_pct = (wins / len(df) * 100).round(1)

print('=== Wins distribution ===')
for m, w, p in zip(wins.index, wins.values, wins_pct.values):
    bar = '█' * int(p / 2)
    print(f'  {m:10s}  {w:3d}/{len(df)}  ({p:4.1f}%)  {bar}')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0' if w > 5 else '#999999' for w in wins.values]
ax.bar(wins.index, wins.values, color=colors)
ax.set_ylabel('Number of datasets where method is best')
ax.set_title('Method Wins Distribution\n(grey = fewer than 5 wins — candidate for removal)')
for i, (m, w) in enumerate(zip(wins.index, wins.values)):
    ax.text(i, w + 0.3, str(w), ha='center', fontsize=10)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_wins.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'\nSaved → {path}')

=== Wins distribution ===
  kmeans       28/94  (29.8%)  ██████████████
  dbscan        5/94  ( 5.3%)  ██
  agg          17/94  (18.1%)  █████████
  gmm          21/94  (22.3%)  ███████████
  autoenc      13/94  (13.8%)  ██████
  dictlearn    10/94  (10.6%)  █████

Saved → c:\MLResearch\outputs\figures\method_wins.png


## Analysis 3: Marginal Contribution to Oracle

For each method m, compute:
`marginal(m) = mean(oracle_LSE_with_all_6) - mean(oracle_LSE_without_m)`

The current oracle mean LSE is **0.7307**. Marginal contributions are GMM +0.0193, agglomerative +0.0101, k-means +0.0092, DBSCAN +0.0079, autoencoder +0.0067, and dictionary learning +0.0053.


In [4]:
oracle_all = df[LSE_COLS].max(axis=1).mean()
print(f'Oracle (all 6 methods): {oracle_all:.4f}')

marginal = {}
for m, col in zip(METHOD_NAMES, LSE_COLS):
    remaining_cols = [c for c in LSE_COLS if c != col]
    oracle_without = df[remaining_cols].max(axis=1).mean()
    marginal[m] = round(oracle_all - oracle_without, 4)

print('\n=== Marginal contribution to oracle ===')
print(f'  (oracle with all 6 = {oracle_all:.4f})')
for m, contrib in sorted(marginal.items(), key=lambda x: x[1], reverse=True):
    tag = ' ← candidate for removal' if contrib < 0.01 else ''
    print(f'  {m:10s}  +{contrib:.4f}{tag}')

fig, ax = plt.subplots(figsize=(8, 4))
names  = list(marginal.keys())
values = list(marginal.values())
colors = ['#4C72B0' if v >= 0.01 else '#DD8452' for v in values]
ax.bar(names, values, color=colors)
ax.axhline(0.01, ls='--', color='red', lw=1, label='0.01 threshold')
ax.set_ylabel('Marginal contribution to oracle (mean LSE)')
ax.set_title('Method Marginal Contribution\n(orange = < 0.01 — candidate for removal)')
ax.legend()
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_marginal.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'\nSaved → {path}')

Oracle (all 6 methods): 0.7307

=== Marginal contribution to oracle ===
  (oracle with all 6 = 0.7307)
  gmm         +0.0193
  agg         +0.0101
  kmeans      +0.0092 ← candidate for removal
  dbscan      +0.0079 ← candidate for removal
  autoenc     +0.0067 ← candidate for removal
  dictlearn   +0.0053 ← candidate for removal

Saved → c:\MLResearch\outputs\figures\method_marginal.png


## Decision

All six pseudo-label generators are kept for the downstream benchmark.

The retention rule removes a method only if it has both low wins and low marginal contribution. Although several methods have marginal contribution below 0.01, every method wins at least 5 datasets, so none are recommended for removal. Proceed to notebook 04 with the full method set.


In [5]:
summary = pd.DataFrame({
    'wins'       : wins,
    'wins_pct'   : wins_pct,
    'marginal'   : pd.Series(marginal),
}).reindex(METHOD_NAMES)

summary['keep'] = (summary['wins'] >= 5) | (summary['marginal'] >= 0.01)

print('=== Method Retention Summary ===')
print(summary.to_string())
print()
print('Methods recommended for removal (fails BOTH thresholds):')
candidates = summary[~summary['keep']].index.tolist()
print(' ', ', '.join(candidates) if candidates else 'None')
print()
print('Note: proceed to 04_metafeatures.ipynb with the chosen method set.')
summary.to_csv(os.path.join(META_DIR, 'method_redundancy_summary.csv'))
print(f'Summary saved → {os.path.join(META_DIR, "method_redundancy_summary.csv")}')

=== Method Retention Summary ===
           wins  wins_pct  marginal  keep
kmeans       28      29.8    0.0092  True
dbscan        5       5.3    0.0079  True
agg          17      18.1    0.0101  True
gmm          21      22.3    0.0193  True
autoenc      13      13.8    0.0067  True
dictlearn    10      10.6    0.0053  True

Methods recommended for removal (fails BOTH thresholds):
  None

Note: proceed to 04_metafeatures.ipynb with the chosen method set.
Summary saved → c:\MLResearch\data\meta_table\method_redundancy_summary.csv
